### Generate functions from an ANN
Input: 

- Input layer size: ILS, int
- Number of hidden layers: NL, int
- Size of layers: LS, int
- Activation functions: activation_functions, list
- Number of functions per activation function: NREP, int

In [10]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.gridspec as gridspec
import pickle

In [11]:
# Network Architecture
ILS = 1 #Input layer size
NL, LS = 5, 10 # Number of hidden layers and their size

# Initialize weights
def init_w(ils=ILS, nl=NL, ls=LS):
    nu = [ils] + [ls] * nl
    w = []
    for l in range(nl):
        w.append([])
        for i in range(nu[l]):
            w[l].append([])
            for j in range(nu[l+1]):
                w[l][i].append(2*(np.random.uniform()-.5)**1)
    w.append([[(2*(np.random.uniform()-.5))**1 for i in range(nu[nl])]])

        
    return w

# Import weights
def import_w(file):
    with open(file, 'rb') as f:
        w = pickle.load(f)
    return w
    
# Compute output
def compute_output(invals, w, activation_fun):
    # Input layer
    cl = np.array(invals)
    # Hidden layers
    for l in range(len(w)-1):
        #print(len(w[l][0]))
        cl = np.array([np.sum([w[l][i][j]*cl[i] for i in range(len(cl))]) for j in range(len(w[l][0]))])
        if activation_fun=='tanh':
            cl = np.tanh(cl)
        elif activation_fun=='ReLU':
            cl=[cl_i if cl_i>0 else 0 for cl_i in cl]
        elif activation_fun=='leaky_ReLU':
            cl=[cl_i if cl_i>0 else 0.01*cl_i for cl_i in cl]
            
        
    return(np.sum([w[-1][i][0] * cl[i] for i in range(len(w[-1]))]))

In [18]:
NREP = 10 #number of functions
xmin=-4;xmax=4;#new_step=0.0125 #limits of x and resolution step to generate functions.0.5x- 0.1, 2x- 0.025, 4x - 0.0125, 1000 points -0.0004

d_all = pd.DataFrame({'x1' : [], 'y': [], 'rep': []})
d_all_raw = pd.DataFrame({'x1' : [], 'y_raw': [], 'rep': []})

activation_functions=['tanh', 'leaky_ReLU'] #tanh, leaky_ReLU
activation_function='leaky_ReLU'


resolutions={'0.5x':'0.1', '1x':'0.05' , '2x': '0.025' , '4e-3x':'0.004' }
resolution='1x' #0.5x, 1x, 2x, 4e-3x
new_step=float(resolutions[resolution])/5 #We take new five points per existing point


#load pickle with weights from neural networks
filename = 'NN_weights_' + activation_function + '_NREP_10_.pickle'
weights = import_w('../../data/generative_data/' + filename)

#load low-resolution non-normalized original data
low_res_data=pd.read_csv('../../data/generative_data/' + 'Non_normalized_NN_function_'+ str(activation_function) + '_NREP_10_res_0.05' +
                          '_data.csv', index_col=0)
#display(low_res_data)

for rep in range(NREP):
    #load pickle with weights
    w=weights[rep]

    #Generate x
    x1 = np.arange(xmin, xmax, new_step)

    #Generate y
    x1s, ys = [], []
    y = [compute_output([1, thisx1], w, activation_function) for thisx1 in x1]
    x1s=x1
    ys=y
    ys = np.array(ys)

    # Normalize y to [0, 1] with lower resolution data
    low_res_rep=low_res_data[low_res_data['rep']==rep]
    ys_low_res=np.array(low_res_rep.y_raw)
    ys_norm = (ys - min(ys_low_res)) / (max(ys_low_res) - min(ys_low_res) + 1e-15) #normalize with maximum and minimum unnormalized values



    #Save normalized high resolution functions to dataframes
    d = pd.DataFrame({'x1' : x1s, 'y' : ys_norm, 'rep': rep})
    #display(d)
    #display(low_res_rep)
    d_all=pd.concat([d_all,d])
    
    #Save non-normalized high resolution version
    d_raw = pd.DataFrame({'x1' : x1s, 'y_raw' : ys, 'rep': rep})
    
    #extract "old" points from raw dataset.
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #d_raw=d_raw.round({'x1':2})
    low_res_rep=low_res_rep.round({'x1':2})
    #d_raw_old_points=d_raw[d_raw['x1'].isin(low_res_rep['x1'].tolist() )]    
    
    #display(low_res_rep)
    #print("old  raw points")
    #display(d_raw_old_points)
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    #extract new normalized points
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #display(d)
    d=d.round({'x1':2})
    #display(low_res_rep['x1'].tolist() )
    #display(d['x1'].tolist() )
    #low_res_rep=low_res_rep.round({'x1':2})
    d_normalized_points=d[d['x1'].isin(low_res_rep['x1'].tolist() )]    
    
    #display(low_res_rep)
    #print("old  raw points")
    display(d_normalized_points)
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    
    #display(d_raw)
    d_all_raw=pd.concat([d_all_raw,d_raw])

#Save to dataframe
save=False
if save==True:
    d_all.to_csv('../../data/generative_data/' + 'NN_function_' + activation_function + '_NREP_' + str(NREP) + '_res_' + str(new_step) + 'interpolation_data.csv')
    d_all_raw.to_csv('../../data/generative_data/' + 'Non_normalized_NN_function_' + activation_function + '_NREP_' + str(NREP) + '_res_' + str(new_step) + 'interpolation_data.csv')

,x1,y,rep
0,-4.00,0.018426,0
5,-3.95,0.018062,0
10,-3.90,0.017699,0
15,-3.85,0.017335,0
20,-3.80,0.016971,0
...,...,...,...
775,3.75,0.956572,0
780,3.80,0.967429,0
785,3.85,0.978286,0
790,3.90,0.989143,0


,x1,y,rep
0,-4.00,1.000000e+00,1
5,-3.95,9.999643e-01,1
10,-3.90,9.999286e-01,1
15,-3.85,9.998929e-01,1
20,-3.80,9.998572e-01,1
...,...,...,...
775,3.75,5.446269e-02,1
780,3.80,4.084702e-02,1
785,3.85,2.723135e-02,1
790,3.90,1.361567e-02,1


,x1,y,rep
0,-4.00,0.000000,2
5,-3.95,0.011186,2
10,-3.90,0.022372,2
15,-3.85,0.033557,2
20,-3.80,0.044743,2
...,...,...,...
775,3.75,0.202385,2
780,3.80,0.186148,2
785,3.85,0.169912,2
790,3.90,0.153675,2


,x1,y,rep
0,-4.00,1.000000e+00,3
5,-3.95,9.999990e-01,3
10,-3.90,9.999980e-01,3
15,-3.85,9.999971e-01,3
20,-3.80,9.999961e-01,3
...,...,...,...
775,3.75,3.602994e-02,3
780,3.80,2.702246e-02,3
785,3.85,1.801497e-02,3
790,3.90,9.007485e-03,3


,x1,y,rep
0,-4.00,1.000000e+00,4
5,-3.95,9.993563e-01,4
10,-3.90,9.987125e-01,4
15,-3.85,9.980688e-01,4
20,-3.80,9.974250e-01,4
...,...,...,...
775,3.75,4.642032e-02,4
780,3.80,3.481524e-02,4
785,3.85,2.321016e-02,4
790,3.90,1.160508e-02,4


,x1,y,rep
0,-4.00,8.658672e-01,5
5,-3.95,8.684451e-01,5
10,-3.90,8.710230e-01,5
15,-3.85,8.736555e-01,5
20,-3.80,8.768476e-01,5
...,...,...,...
775,3.75,7.032480e-02,5
780,3.80,5.274360e-02,5
785,3.85,3.516240e-02,5
790,3.90,1.758120e-02,5


,x1,y,rep
0,-4.00,0.435815,6
5,-3.95,0.426845,6
10,-3.90,0.417875,6
15,-3.85,0.408905,6
20,-3.80,0.399935,6
...,...,...,...
775,3.75,0.956435,6
780,3.80,0.967326,6
785,3.85,0.978218,6
790,3.90,0.989109,6


,x1,y,rep
0,-4.00,1.000000,7
5,-3.95,0.987289,7
10,-3.90,0.974239,7
15,-3.85,0.961126,7
20,-3.80,0.948013,7
...,...,...,...
775,3.75,0.922379,7
780,3.80,0.933994,7
785,3.85,0.945271,7
790,3.90,0.955012,7


,x1,y,rep
0,-4.00,0.977514,8
5,-3.95,0.959611,8
10,-3.90,0.941708,8
15,-3.85,0.923805,8
20,-3.80,0.906174,8
...,...,...,...
775,3.75,0.950228,8
780,3.80,0.962896,8
785,3.85,0.975565,8
790,3.90,0.987990,8


,x1,y,rep
0,-4.00,7.419524e-01,9
5,-3.95,7.458970e-01,9
10,-3.90,7.498416e-01,9
15,-3.85,7.537862e-01,9
20,-3.80,7.577308e-01,9
...,...,...,...
775,3.75,5.094657e-02,9
780,3.80,3.820993e-02,9
785,3.85,2.547328e-02,9
790,3.90,1.273664e-02,9


In [ ]:
#PLOT -2,2

#Figure Size                                                                                                                                                                                                
cm = 1/2.54  # centimeters in inches                                                                                                                                                                        
width=40*cm;height=8*cm #Width and height of plots 
matplotlib.rcParams['figure.figsize'] = [width, height]

rows=2;cols=5
gs=gridspec.GridSpec(rows,cols)
gs.update(left=0.1,right=0.99,bottom=0.08,top=0.97,wspace=0.5,hspace=0.0)

#Plot train rank (-2,2)
h=0
for r in range(rows):
    for c in range(cols):
        ax_rc=plt.subplot(gs[r,c])
        d=d_all[d_all['rep']==h]
        ax_rc=sns.lineplot(data=d, x='x1', y='y')
        ax_rc.set_xlim(-2,2)
        h+=1
plt.savefig('../../results/seminal_data/' + 'NN_function_high_res_step_' +str(new_step) + '_' + activation_function + '_ILS%d_NL%d_LS%d'  %(ILS, NL,  LS) + '.png', dpi=300)


In [ ]:
#PLOT FULL RANK (-4,4)

#Figure Size                                                                                                                                                                                                
cm = 1/2.54  # centimeters in inches                                                                                                                                                                        
width=40*cm;height=8*cm #Width and height of plots 
matplotlib.rcParams['figure.figsize'] = [width, height]


h=0
for r in range(rows):
    for c in range(cols):
        ax_rc=plt.subplot(gs[r,c])
        d=d_all[d_all['rep']==h]
        ax_rc=sns.lineplot(data=d, x='x1', y='y')
        ax_rc.vlines(x=[-2, 2], ymin=0, ymax=1, color='k',linestyle='--')
        ax_rc.set_xlim(-4,4)
        h+=1
plt.savefig('../../results/seminal_data/' + 'NN_function_high_res_step_' +str(new_step) + '_' + activation_function + '_ILS%d_NL%d_LS%d_full_rank'  %(ILS, NL,  LS) + '.png', dpi=300)